# 97 — Compare Geode and nodal common shots

Builds a comparison index for common Geode/nodal shots and quick-look plots of stacked nodal gathers. Geode waveform reading/conversion is kept diagnostic because file formats may vary.

Outputs/replaces:
- `geode_nodal_common_shot_comparisons`
- `geode_nodal_comparison_files`

In [1]:
from pathlib import Path
import sqlite3
import json
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import read, Stream, Trace, UTCDateTime

PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"
CATALOG_DB.parent.mkdir(parents=True, exist_ok=True)
print("CATALOG_DB:", CATALOG_DB)

OUT_ROOT = PROJECT_ROOT / "geode_nodal_common_shot_comparisons_v1"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
COMPONENT = "Z"
TARGET_SURVEYS = None
print("OUT_ROOT:", OUT_ROOT)

CATALOG_DB: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
OUT_ROOT: /Volumes/tachyon/LBSSP_DATA/geode_nodal_common_shot_comparisons_v1


## 1. Load tables safely

In [2]:
REQUIRED = ["geode_events", "nodal_stacks", "nodal_stack_files"]
OWNED = ["geode_nodal_common_shot_comparisons", "geode_nodal_comparison_files"]

with sqlite3.connect(CATALOG_DB) as conn:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)["name"].tolist()
missing = [t for t in REQUIRED if t not in tables]
if missing:
    raise RuntimeError(f"Missing inputs: {missing}. Run 92 and 95 first.")

conn = sqlite3.connect(CATALOG_DB)
geode_events = pd.read_sql("SELECT * FROM geode_events", conn)
nodal_stacks = pd.read_sql("SELECT * FROM nodal_stacks", conn)
nodal_stack_files = pd.read_sql("SELECT * FROM nodal_stack_files", conn)

if TARGET_SURVEYS is not None:
    geode_events = geode_events[geode_events["survey"].astype(str).isin(TARGET_SURVEYS)].copy()
    nodal_stacks = nodal_stacks[nodal_stacks["geode_survey"].astype(str).isin(TARGET_SURVEYS)].copy()

display(nodal_stacks.head())

,stack_id,geode_event_id,geode_survey,line,file_no,source_x_truth_m,source_type,n_candidate_members,n_accepted_members,reference_nodal_event_id,median_xcorr_shift_s,median_xcorr_corrcoef,min_member_time_from_final_s,max_member_time_from_final_s,output_dir,status
0,NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,GEODE_T1_1M_REFRACTION_F3006,T1_1m_refraction,T1,3006,84.5,hammer,6,4,T1_N2_Refraction1m_T1_N2_E00008,-0.008,0.928603,-46.990,-0.286,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,ok
1,NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,GEODE_T1_1M_REFRACTION_F3008,T1_1m_refraction,T1,3008,88.5,hammer,3,2,T1_N2_Refraction1m_T1_N2_E00011,0.003,0.976702,-21.410,-7.324,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,ok
2,NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,GEODE_T1_1M_REFRACTION_F3009,T1_1m_refraction,T1,3009,90.5,hammer,7,7,T1_N2_Refraction1m_T1_N2_E00020,0.026,0.929131,-42.758,0.060,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,ok
3,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,GEODE_T1_1M_REFRACTION_F3011,T1_1m_refraction,T1,3011,94.5,hammer,6,3,T1_N2_Refraction1m_T1_N2_E00046,0.056,0.984687,-46.552,-0.332,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,ok
4,NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,GEODE_T1_1M_REFRACTION_F3012,T1_1m_refraction,T1,3012,96.5,hammer,5,3,T1_N2_Refraction1m_T1_N2_E00054,0.013,0.942910,-29.534,-0.174,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,ok


## 2. Build comparison index

In [3]:
nodal_mseed = nodal_stack_files[
    (nodal_stack_files["file_type"].astype(str).eq("mseed")) &
    (nodal_stack_files["component"].astype(str).eq(COMPONENT))
].copy()

rows = []
for _, ns in nodal_stacks.iterrows():
    geode_event_id = ns["geode_event_id"]
    ge = geode_events[geode_events["geode_event_id"].astype(str).eq(str(geode_event_id))]
    if ge.empty:
        continue
    ge = ge.iloc[0]
    nf = nodal_mseed[nodal_mseed["stack_id"].astype(str).eq(str(ns["stack_id"]))]
    nodal_path = nf.iloc[0]["file_path"] if len(nf) else None
    geode_path = ge.get("geode_file_path")

    geode_exists = Path(str(geode_path)).exists() if geode_path not in [None, "None", "nan"] else False
    nodal_exists = Path(str(nodal_path)).exists() if nodal_path not in [None, "None", "nan"] else False

    rows.append({
        "comparison_id": f"CMP_{ns['stack_id']}",
        "stack_id": ns["stack_id"],
        "geode_event_id": geode_event_id,
        "survey": ns["geode_survey"],
        "line": ns["line"],
        "file_no": ns["file_no"],
        "source_x_m": ns["source_x_truth_m"],
        "component": COMPONENT,
        "nodal_stack_mseed_path": nodal_path,
        "nodal_stack_exists": nodal_exists,
        "geode_file_path": geode_path,
        "geode_file_exists": geode_exists,
        "geode_read_format": ge.get("geode_read_format"),
        "geode_n_traces": ge.get("geode_n_traces"),
        "status": "ready" if nodal_exists and geode_exists else "missing_file",
    })

comparisons = pd.DataFrame(rows)
display(comparisons["status"].value_counts(dropna=False) if len(comparisons) else pd.Series(dtype=int))
display(comparisons.head())

status
ready           84
missing_file    66
Name: count, dtype: int64

,comparison_id,stack_id,geode_event_id,survey,line,file_no,source_x_m,component,nodal_stack_mseed_path,nodal_stack_exists,geode_file_path,geode_file_exists,geode_read_format,geode_n_traces,status
0,CMP_NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,GEODE_T1_1M_REFRACTION_F3006,T1_1m_refraction,T1,3006,84.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,True,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,True,SEG2,72.0,ready
1,CMP_NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,GEODE_T1_1M_REFRACTION_F3008,T1_1m_refraction,T1,3008,88.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,True,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,True,SEG2,72.0,ready
2,CMP_NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,GEODE_T1_1M_REFRACTION_F3009,T1_1m_refraction,T1,3009,90.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,True,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,True,SEG2,72.0,ready
3,CMP_NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,GEODE_T1_1M_REFRACTION_F3011,T1_1m_refraction,T1,3011,94.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,True,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,True,SEG2,72.0,ready
4,CMP_NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,GEODE_T1_1M_REFRACTION_F3012,T1_1m_refraction,T1,3012,96.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,True,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,True,SEG2,72.0,ready


## 3. Quick-look plots and safe write

In [4]:
file_rows = []

def quick_plot_nodal(row):
    p = Path(row["nodal_stack_mseed_path"])
    st = read(str(p))
    fig, ax = plt.subplots(figsize=(10, 5))
    traces = [tr for tr in st if hasattr(tr.stats, "receiver_x_m")]
    if not traces:
        traces = list(st)
    for i, tr in enumerate(traces[:100]):
        y = tr.data.astype(float)
        mx = np.nanmax(np.abs(y))
        if np.isfinite(mx) and mx > 0:
            y = y / mx
        t = tr.stats.starttime - UTCDateTime(0) + np.arange(tr.stats.npts) * tr.stats.delta
        x = getattr(tr.stats, "receiver_x_m", i)
        ax.plot(x + 0.5*y, t, linewidth=0.5)
    ax.invert_yaxis()
    ax.set_title(f"{row['survey']} F{row['file_no']} x={row['source_x_m']} m\nNodal stack; Geode file exists={row['geode_file_exists']}")
    ax.set_xlabel("Receiver x or trace index")
    ax.set_ylabel("Time (s)")
    ax.grid(True, alpha=0.3)
    out = OUT_ROOT / str(row["line"]) / str(row["survey"]) / f"{row['comparison_id']}_quicklook.png"
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=180)
    plt.close(fig)
    return out

for _, row in comparisons.iterrows():
    if row["nodal_stack_exists"]:
        try:
            out = quick_plot_nodal(row)
            file_rows.append({
                "comparison_id": row["comparison_id"],
                "stack_id": row["stack_id"],
                "geode_event_id": row["geode_event_id"],
                "file_type": "png_quicklook",
                "file_path": str(out),
            })
        except Exception as e:
            print("plot failed", row["comparison_id"], e)

comparison_files = pd.DataFrame(file_rows)
if comparison_files is None or len(comparison_files.columns) == 0:
    comparison_files = pd.DataFrame(columns=["comparison_id", "stack_id", "geode_event_id", "file_type", "file_path"])

with sqlite3.connect(CATALOG_DB) as conn:
    for tname in OWNED:
        conn.execute(f'DROP TABLE IF EXISTS "{tname}"')
    comparisons.to_sql("geode_nodal_common_shot_comparisons", conn, if_exists="fail", index=False)
    comparison_files.to_sql("geode_nodal_comparison_files", conn, if_exists="fail", index=False)
    conn.commit()

comparisons.to_csv(OUT_ROOT / "geode_nodal_common_shot_comparisons.csv", index=False)
comparison_files.to_csv(OUT_ROOT / "geode_nodal_comparison_files.csv", index=False)
print("Wrote 97-owned tables:", OWNED)

Wrote 97-owned tables: ['geode_nodal_common_shot_comparisons', 'geode_nodal_comparison_files']
